## Sin Transfer Learning
### *K-fold*

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
from collections import defaultdict, Counter
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import wandb
from sklearn.model_selection import KFold

In [2]:
with open("config.json", "r") as f:
    config = json.load(f)
DATASET_PATH = config["DATASET_PATH"]

In [3]:
class ChestXrayDataset3Clases(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        labels_map = {"NORMAL": 0, "BACTERIA": 1, "VIRUS": 2}
        for folder, label in labels_map.items():
            folder_path = os.path.join(root_dir, "PNEUMONIA") if folder != "NORMAL" else os.path.join(root_dir, "NORMAL")
            if folder != "NORMAL":
                folder_path = os.path.join(folder_path, folder)
            if not os.path.exists(folder_path):
                continue
            for root, _, files in os.walk(folder_path):
                for file in files:
                    if file.lower().endswith(('.jpeg', '.jpg', '.png')):
                        self.samples.append((os.path.join(root, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(256, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std =[0.229, 0.224, 0.225])
])

In [6]:
class TransformedSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [7]:
train_data = ChestXrayDataset3Clases(os.path.join(DATASET_PATH, "train"), transform=None)
val_data = ChestXrayDataset3Clases(os.path.join(DATASET_PATH, "val"), transform=None)
full_data = torch.utils.data.ConcatDataset([train_data, val_data])

test_data  = ChestXrayDataset3Clases(os.path.join(DATASET_PATH, "test"), val_transform)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    try:
        import torch_directml
        device = torch_directml.device()
    except ImportError:
        device = torch.device("cpu")

In [9]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256*16*16, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

Este modelo SimpleCNN es una red convolucional diseñada para clasificación de imágenes en tres clases. La parte de features aplica varias capas convolucionales con batch normalization, activación ReLU y max pooling para extraer representaciones jerárquicas de la imagen. Después, la salida se aplana y pasa por un bloque fully connected con capas lineales, activaciones ReLU y dropout para reducir sobreajuste, finalizando en una capa lineal que produce las predicciones de clase.

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = {}

all_train_losses, all_val_losses = [], []
all_train_accs, all_val_accs = [], []

for fold, (train_idx, val_idx) in enumerate(kf.split(full_data)):

    wandb.init(
        project="chest_xray_simplecnn",
        name=f"Fold_{fold + 1}",
        config={
            "epochs": 35,
            "batch_size": 64,
            "learning_rate": 1e-4,
            "weight_decay": 0.0016,
            "architecture": "SimpleCNN",
            "k_folds": 5
        },
        reinit=True
    )

    print(f"\n--Fold {fold+1}--")

    train_subset = Subset(full_data, train_idx)
    val_subset   = Subset(full_data, val_idx)

    train_subset = TransformedSubset(train_subset, transform=train_transform)
    val_subset   = TransformedSubset(val_subset, transform=val_transform)

    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=32, shuffle=False)

    model = SimpleCNN(num_classes=3).to(device)

    labels = [full_data[i][1] for i in train_idx]
    counts = Counter(labels)
    total = sum(counts.values())
    weights = torch.tensor(
        [total / (3 * counts[i]) if counts[i] > 0 else 0 for i in range(3)],
        dtype=torch.float
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.016)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=0.0016)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=6,
        threshold=0.002,
        min_lr=1e-6
    )

    best_val_loss = float('inf')
    patience_loss = 10
    patience_counter_loss = 0

    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    for epoch in range(35):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total
        scheduler.step(val_loss)

        print(f"Fold {fold+1}, Epoch {epoch+1}: Train Loss {train_loss:.4f} "
              f"Train Acc {train_acc:.2f}% Val Loss {val_loss:.4f} Val Acc {val_acc:.2f}%")

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)

        all_train_losses.append(train_loss)
        all_val_losses.append(val_loss)
        all_train_accs.append(train_acc)
        all_val_accs.append(val_acc)

        wandb.log({
            "epoch": epoch,
            "fold": fold + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        # Early stopping
        if val_loss < best_val_loss - 1e-3:
            best_val_loss = val_loss
            patience_counter_loss = 0
            torch.save(model.state_dict(), f"best_simplecnn_fold{fold + 1}.pth")
        else:
            patience_counter_loss += 1
            if patience_counter_loss >= patience_loss:
                print("Early stopping activado (criterio: val_loss)")
                break

    results[fold] = best_val_loss
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label="Train Loss", marker='o')
    plt.plot(val_losses, label="Val Loss", marker='o')
    plt.title(f"Loss por época (Fold {fold+1})")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label="Train Acc", marker='o')
    plt.plot(val_accs, label="Val Acc", marker='o')
    plt.title(f"Accuracy por época (Fold {fold+1})")
    plt.xlabel("Época")
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.tight_layout()

    wandb.log({f"loss_acc_fold{fold+1}": wandb.Image(plt)})
    plt.close()

    model.eval()
    classes = ["NORMAL", "BACTERIA", "VIRUS"]
    all_labels, all_preds = [], []
    class_correct = defaultdict(int)
    class_total = defaultdict(int)
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            for label, pred in zip(labels, predicted):
                class_total[classes[label]] += 1
                if label == pred:
                    class_correct[classes[label]] += 1

    accuracy = 100 * correct / total
    print(f"\nFold {fold+1} - Accuracy en test: {accuracy:.2f}%")
    print("Accuracy por clase:")
    for c in classes:
        acc = 100 * class_correct[c] / class_total[c] if class_total[c] > 0 else 0
        print(f"{c}: {acc:.2f}%")

    wandb.log({f"test_accuracy_fold{fold+1}": accuracy})

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Matriz de Confusión (Fold {fold+1})")
    plt.tight_layout()

    wandb.log({f"confusion_matrix_fold{fold+1}": wandb.Image(plt)})
    plt.close()


wandb: Currently logged in as: judporper (judporper-university-of-las-palmas-de-gran-canaria) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



--Fold 1--
Fold 1, Epoch 1: Train Loss 1.0185 Train Acc 49.03% Val Loss 0.7728 Val Acc 67.53%
Fold 1, Epoch 2: Train Loss 0.8344 Train Acc 57.66% Val Loss 0.7550 Val Acc 65.62%
Fold 1, Epoch 3: Train Loss 0.7745 Train Acc 60.22% Val Loss 0.8099 Val Acc 65.71%
Fold 1, Epoch 4: Train Loss 0.7557 Train Acc 61.46% Val Loss 0.6773 Val Acc 72.97%
Fold 1, Epoch 5: Train Loss 0.7462 Train Acc 62.72% Val Loss 0.8356 Val Acc 64.57%
Fold 1, Epoch 6: Train Loss 0.7223 Train Acc 65.35% Val Loss 0.7883 Val Acc 66.28%
Fold 1, Epoch 7: Train Loss 0.7204 Train Acc 63.89% Val Loss 0.6333 Val Acc 74.50%
Fold 1, Epoch 8: Train Loss 0.7118 Train Acc 65.54% Val Loss 0.6743 Val Acc 71.73%
Fold 1, Epoch 9: Train Loss 0.7053 Train Acc 66.05% Val Loss 0.7026 Val Acc 67.53%
Fold 1, Epoch 10: Train Loss 0.6943 Train Acc 65.88% Val Loss 0.6243 Val Acc 74.88%
Fold 1, Epoch 11: Train Loss 0.6873 Train Acc 67.93% Val Loss 0.6298 Val Acc 74.59%
Fold 1, Epoch 12: Train Loss 0.6920 Train Acc 67.81% Val Loss 0.6436 Val 

epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
fold,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_fold1,▁
train_acc,▁▃▄▄▅▆▅▆▆▆▆▆▆▇▆▇▆▇█▇▇█▇████████████
train_loss,█▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁
val_acc,▃▂▂▅▁▂▆▅▃▆▆▆▃▆▅▃▅▇▇▆▇▇▇█▇█▇███▇▇███
val_loss,▆▆▇▄█▇▃▄▅▂▃▃▆▃▃█▆▂▂▃▂▃▂▂▂▁▂▁▁▁▃▂▁▁▁
epoch,34
fold,1
test_accuracy_fold1,84.77564
train_acc,73.57228



--Fold 2--
Fold 2, Epoch 1: Train Loss 1.0404 Train Acc 48.34% Val Loss 0.8239 Val Acc 47.95%
Fold 2, Epoch 2: Train Loss 0.8406 Train Acc 56.89% Val Loss 0.7283 Val Acc 70.87%
Fold 2, Epoch 3: Train Loss 0.8005 Train Acc 59.28% Val Loss 0.7109 Val Acc 72.02%
Fold 2, Epoch 4: Train Loss 0.7745 Train Acc 61.31% Val Loss 0.7379 Val Acc 52.24%
Fold 2, Epoch 5: Train Loss 0.7425 Train Acc 63.61% Val Loss 0.6743 Val Acc 68.39%
Fold 2, Epoch 6: Train Loss 0.7481 Train Acc 62.17% Val Loss 0.6920 Val Acc 63.04%
Fold 2, Epoch 7: Train Loss 0.7214 Train Acc 64.16% Val Loss 0.7360 Val Acc 56.92%
Fold 2, Epoch 8: Train Loss 0.7114 Train Acc 64.44% Val Loss 0.6951 Val Acc 60.08%
Fold 2, Epoch 9: Train Loss 0.6961 Train Acc 65.78% Val Loss 0.7279 Val Acc 66.09%
Fold 2, Epoch 10: Train Loss 0.6903 Train Acc 66.05% Val Loss 0.6464 Val Acc 68.77%
Fold 2, Epoch 11: Train Loss 0.6972 Train Acc 66.36% Val Loss 0.6202 Val Acc 74.02%
Fold 2, Epoch 12: Train Loss 0.6852 Train Acc 66.12% Val Loss 0.6810 Val 

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
fold,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_fold2,▁
train_acc,▁▃▄▅▅▅▅▅▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇█▇███
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▁▂▂▁▁▁▁▁▁▁
val_acc,▁▆▆▂▆▄▃▄▅▆▇▇▆▅▇▇▆▇███▇▇▇█▇████
val_loss,▇▅▄▅▃▄▅▄▅▃▂▄▃█▂▁▃▁▂▁▁▃▄▂▂▂▂▁▂▁
epoch,29
fold,2
test_accuracy_fold2,82.53205
train_acc,73.11828



--Fold 3--
Fold 3, Epoch 1: Train Loss 1.0216 Train Acc 49.19% Val Loss 0.8066 Val Acc 58.60%
Fold 3, Epoch 2: Train Loss 0.8365 Train Acc 56.45% Val Loss 0.7280 Val Acc 67.02%
Fold 3, Epoch 3: Train Loss 0.8014 Train Acc 59.13% Val Loss 0.7011 Val Acc 68.07%
Fold 3, Epoch 4: Train Loss 0.7713 Train Acc 60.89% Val Loss 0.7193 Val Acc 68.74%
Fold 3, Epoch 5: Train Loss 0.7501 Train Acc 62.06% Val Loss 0.7099 Val Acc 71.03%
Fold 3, Epoch 6: Train Loss 0.7370 Train Acc 62.76% Val Loss 0.6322 Val Acc 73.33%
Fold 3, Epoch 7: Train Loss 0.7275 Train Acc 63.55% Val Loss 0.6815 Val Acc 63.48%
Fold 3, Epoch 8: Train Loss 0.7123 Train Acc 64.43% Val Loss 0.6373 Val Acc 74.95%
Fold 3, Epoch 9: Train Loss 0.7120 Train Acc 64.76% Val Loss 0.6382 Val Acc 74.09%
Fold 3, Epoch 10: Train Loss 0.6933 Train Acc 65.50% Val Loss 0.6547 Val Acc 73.42%
Fold 3, Epoch 11: Train Loss 0.6984 Train Acc 66.34% Val Loss 0.6698 Val Acc 74.38%
Fold 3, Epoch 12: Train Loss 0.6933 Train Acc 65.77% Val Loss 0.6314 Val 

epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
fold,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_fold3,▁
train_acc,▁▃▄▅▅▅▆▆▆▆▇▆▇███
train_loss,█▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁
val_acc,▁▅▅▅▆▇▃██▇██▇█▇█
val_loss,█▅▄▅▄▁▃▁▁▂▃▁▄▂▃▂
epoch,15
fold,3
test_accuracy_fold3,80.12821
train_acc,70.52078



--Fold 4--
Fold 4, Epoch 1: Train Loss 0.9880 Train Acc 49.12% Val Loss 0.7914 Val Acc 66.06%
Fold 4, Epoch 2: Train Loss 0.8262 Train Acc 57.26% Val Loss 0.7534 Val Acc 70.46%
Fold 4, Epoch 3: Train Loss 0.7877 Train Acc 60.77% Val Loss 0.7543 Val Acc 69.98%
Fold 4, Epoch 4: Train Loss 0.7710 Train Acc 59.70% Val Loss 0.7404 Val Acc 71.51%
Fold 4, Epoch 5: Train Loss 0.7244 Train Acc 63.19% Val Loss 0.8026 Val Acc 67.02%
Fold 4, Epoch 6: Train Loss 0.7131 Train Acc 63.43% Val Loss 0.7080 Val Acc 71.89%
Fold 4, Epoch 7: Train Loss 0.6957 Train Acc 65.72% Val Loss 0.7082 Val Acc 70.08%
Fold 4, Epoch 8: Train Loss 0.6962 Train Acc 64.45% Val Loss 0.8637 Val Acc 58.89%
Fold 4, Epoch 9: Train Loss 0.6814 Train Acc 66.17% Val Loss 0.7582 Val Acc 71.03%
Fold 4, Epoch 10: Train Loss 0.6730 Train Acc 67.70% Val Loss 0.7503 Val Acc 68.45%
Fold 4, Epoch 11: Train Loss 0.6696 Train Acc 67.20% Val Loss 0.7340 Val Acc 71.22%
Fold 4, Epoch 12: Train Loss 0.6857 Train Acc 67.58% Val Loss 0.6997 Val 

epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
fold,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_fold4,▁
train_acc,▁▃▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▆▇▇▆▇▇▇▇▇▇▇██▇█▇██
train_loss,█▅▅▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val_acc,▄▅▅▅▄▆▅▁▅▄▅▆▆▇▆▆▇▇▆▇▅▆▆▆▇▇███▇▇▇███
val_loss,▆▅▅▅▇▄▄█▆▅▅▄▅▃▃▃▂▂▆▃▆▃▃▅▃▁▂▁▂▂▂▃▂▁▁
epoch,34
fold,4
test_accuracy_fold4,83.8141
train_acc,74.58194



--Fold 5--
Fold 5, Epoch 1: Train Loss 1.0265 Train Acc 47.49% Val Loss 0.7661 Val Acc 68.36%
Fold 5, Epoch 2: Train Loss 0.8424 Train Acc 56.69% Val Loss 0.7398 Val Acc 70.75%
Fold 5, Epoch 3: Train Loss 0.8020 Train Acc 58.89% Val Loss 0.7298 Val Acc 71.51%
Fold 5, Epoch 4: Train Loss 0.7666 Train Acc 60.37% Val Loss 0.6652 Val Acc 72.47%
Fold 5, Epoch 5: Train Loss 0.7553 Train Acc 62.16% Val Loss 0.6821 Val Acc 71.80%
Fold 5, Epoch 6: Train Loss 0.7388 Train Acc 61.75% Val Loss 0.6354 Val Acc 73.52%
Fold 5, Epoch 7: Train Loss 0.7245 Train Acc 62.80% Val Loss 0.6739 Val Acc 64.63%
Fold 5, Epoch 8: Train Loss 0.7173 Train Acc 63.38% Val Loss 0.5998 Val Acc 77.25%
Fold 5, Epoch 9: Train Loss 0.6994 Train Acc 64.38% Val Loss 0.6170 Val Acc 77.82%
Fold 5, Epoch 10: Train Loss 0.6848 Train Acc 67.03% Val Loss 0.7036 Val Acc 73.61%
Fold 5, Epoch 11: Train Loss 0.6950 Train Acc 65.70% Val Loss 0.6929 Val Acc 73.04%
Fold 5, Epoch 12: Train Loss 0.6858 Train Acc 66.67% Val Loss 0.6286 Val 

El entrenamiento con validación cruzada en cinco folds muestra un desempeño bastante consistente del modelo SimpleCNN. En el Fold 1 se alcanzó un 84.78% de accuracy en test, con gran rendimiento en la clase BACTERIA. El Fold 2 fue más irregular, con early stopping en la época 30 y un 82.53% en test, manteniendo buen equilibrio aunque con menor precisión en VIRUS. El Fold 3 también terminó antes por early stopping, logrando un 80.13% en test, con BACTERIA fuerte y VIRUS más débil. El Fold 4 mostró una evolución estable hasta el final, alcanzando 83.81% en test y una precisión muy alta en BACTERIA. Finalmente, el Fold 5 fue el más sólido, con una mejora progresiva y un 85.58% en test, destacando un buen rendimiento en todas las clases, especialmente NORMAL y BACTERIA. En conjunto, los cinco folds confirman que el modelo generaliza bien, con resultados entre 80% y 86% de accuracy y una tendencia clara a clasificar mejor los casos de neumonía bacteriana que los virales o normales.